In [ ]:
from __future__ import annotations
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from typing import Optional, Tuple, Dict, Any

class FinancialTradingEnv(gym.Env):
    """
    A simple single-asset trading environment for Gymnasium.

    Observation:
        - A rolling window of the last `window_size` days of [close, volume]
          shaped (window_size, 2), dtype float32.
        - You can optionally standardize features using a rolling z-score
          or global z-score. By default, no scaling.

    Actions (Discrete(3)):
        0 = hold (do-nothing)
        1 = buy  (increase position by +1 up to max_position)
        2 = sell (decrease position by -1 down to -max_position)

    Position:
        - Integer position in [-max_position, ..., 0, ..., +max_position]
        - Each step, unrealized PnL is position * price_change.

    Reward:
        - Reward_t = position_{t-1} * (close_t - close_{t-1})
          - transaction_cost * |position_t - position_{t-1}|
        - I.e., you pay a fixed cost when you change position.

    Episode Termination:
        - When we reach the end of the provided data.

    Info dict:
        {
          'step': int,
          'date': pd.Timestamp or None,
          'position': int,
          'price': float,
          'reward': float,
          'pnl_cum': float
        }

    Notes:
        - Data must provide columns ['close', 'volume'] with a DateTimeIndex or any index.
        - The environment starts at index `window_size`, so the first observation
          contains the previous `window_size` rows.
    """

    metadata = {"render_modes": ["human"], "render_fps": 4}

    def __init__(
        self,
        data: pd.DataFrame,
        window_size: int = 180,
        max_position: int = 1,
        transaction_cost: float = 0.0,
        standardize: Optional[str] = None,  # None | 'global' | 'rolling'
        render_mode: Optional[str] = None,
        seed: Optional[int] = None,
    ) -> None:
        assert {"close", "volume"}.issubset(set(data.columns)), (
            "data must have columns: 'close' and 'volume'"
        )
        assert window_size >= 2, "window_size must be >= 2"
        assert max_position >= 1, "max_position must be >= 1"
        assert standardize in (None, "global", "rolling"), (
            "standardize must be None, 'global', or 'rolling'"
        )

        super().__init__()
        self._rng = np.random.default_rng(seed)

        # Clean and store data
        self.data = data[["close", "volume"]].copy()
        self.data = self.data.replace([np.inf, -np.inf], np.nan).dropna()
        self.close = self.data["close"].to_numpy(dtype=np.float64)
        self.volume = self.data["volume"].to_numpy(dtype=np.float64)
        self.index = self.data.index
        self.n = len(self.data)

        if self.n <= window_size:
            raise ValueError("Not enough data rows to form a single observation window.")

        self.window_size = int(window_size)
        self.max_position = int(max_position)
        self.transaction_cost = float(transaction_cost)
        self.standardize = standardize
        self.render_mode = render_mode

        # Precompute global z-score parameters if needed
        if self.standardize == "global":
            self._mu = self.data.mean(numeric_only=True)
            self._sigma = self.data.std(ddof=0, numeric_only=True).replace(0.0, 1.0)
        else:
            self._mu = None
            self._sigma = None

        # Gym spaces
        self.action_space = spaces.Discrete(3)  # hold, buy, sell
        obs_low = np.full((self.window_size, 2), -np.inf, dtype=np.float32)
        obs_high = np.full((self.window_size, 2), np.inf, dtype=np.float32)
        self.observation_space = spaces.Box(low=obs_low, high=obs_high, dtype=np.float32)

        # State vars
        self._t: int = 0               # current time index in data
        self._position: int = 0        # current integer position
        self._pnl_cum: float = 0.0     # cumulative PnL
        self._last_price: float = np.nan
        self._step_count: int = 0

    def _scale_features(self, window: pd.DataFrame) -> np.ndarray:
        if self.standardize is None:
            return window.to_numpy(dtype=np.float32)
        if self.standardize == "global":
            z = (window - self._mu) / self._sigma
            return z.to_numpy(dtype=np.float32)
        # rolling standardization within the window (per feature)
        mu = window.mean()
        sigma = window.std(ddof=0).replace(0.0, 1.0)
        z = (window - mu) / sigma
        return z.to_numpy(dtype=np.float32)

    def _get_observation(self) -> np.ndarray:
        start = self._t - self.window_size
        end = self._t
        window_df = self.data.iloc[start:end]
        obs = self._scale_features(window_df)
        return obs

    def reset(self, *, seed: Optional[int] = None, options: Optional[Dict[str, Any]] = None) -> Tuple[np.ndarray, Dict[str, Any]]:
        super().reset(seed=seed)
        if seed is not None:
            self._rng = np.random.default_rng(seed)

        # Start at a deterministic point unless 'options' specifies otherwise
        # t is the index of the *next* price we will transition to on step()
        self._t = self.window_size
        self._position = 0
        self._pnl_cum = 0.0
        self._step_count = 0
        self._last_price = float(self.close[self._t - 1])

        obs = self._get_observation()
        info = {
            "step": self._step_count,
            "date": self.index[self._t - 1] if hasattr(self.index, "__len__") else None,
            "position": self._position,
            "price": float(self._last_price),
            "reward": 0.0,
            "pnl_cum": float(self._pnl_cum),
        }
        return obs, info

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, bool, Dict[str, Any]]:
        assert self.action_space.contains(action), f"Invalid action {action}"
        if self._t >= self.n:
            raise RuntimeError("Episode is done. Call reset().")

        prev_position = self._position
        # Map action to position change
        if action == 1:  # buy
            self._position = min(self._position + 1, self.max_position)
        elif action == 2:  # sell
            self._position = max(self._position - 1, -self.max_position)
        # else action == 0 -> hold

        # Price movement from t-1 to t
        price_prev = float(self.close[self._t - 1])
        price_now = float(self.close[self._t])
        price_change = price_now - price_prev

        # PnL based on *previous* position (common convention)
        reward = prev_position * price_change

        # Transaction cost when changing position
        position_change = abs(self._position - prev_position)
        if position_change > 0 and self.transaction_cost > 0:
            reward -= self.transaction_cost * position_change

        self._pnl_cum += reward
        self._step_count += 1
        self._last_price = price_now

        # advance time
        self._t += 1

        # Observations use the last `window_size` rows ending at t-1 (just observed time)
        obs = self._get_observation()

        terminated = self._t >= self.n  # reached end of data
        truncated = False  # no time limit truncation here; add if you like

        info = {
            "step": self._step_count,
            "date": self.index[self._t - 1] if self._t - 1 < len(self.index) else None,
            "position": self._position,
            "price": float(self._last_price),
            "reward": float(reward),
            "pnl_cum": float(self._pnl_cum),
        }

        if self.render_mode == "human":
            self.render()

        return obs.astype(np.float32), float(reward), terminated, truncated, info

    def render(self) -> None:
        print(
            f"t={self._t:5d} | price={self._last_price:,.4f} | pos={self._position:+d} | "
            f"cumPnL={self._pnl_cum:,.4f}"
        )

    def close(self) -> None:
        pass


# -----------------------------
# Example usage (remove in prod)
# -----------------------------
if __name__ == "__main__":
    import yfinance as yf 

    df = yf.download("O", "2020-01-01", "2025-07-01")
    # Create some dummy data
    #dates = pd.date_range("2020-01-01", periods=1000, freq="D")
    #close = np.cumsum(np.random.normal(0, 1, size=1000)) + 100
    #volume = np.random.lognormal(mean=12, sigma=0.5, size=1000)
    #df = pd.DataFrame({"close": close, "volume": volume}, index=dates)

    env = FinancialTradingEnv(
        df,
        window_size=180,
        max_position=1,
        transaction_cost=0.001,
        standardize=None,
        render_mode="human",
        seed=42,
    )

    obs, info = env.reset()
    done = False
    total_reward = 0.0
    while True:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break
    print("Episode finished. Total reward:", total_reward)


[*********************100%***********************]  1 of 1 completed


AssertionError: data must have columns: 'Close' and 'Volume'

In [11]:
df = yf.download("O", "2020-01-01", "2025-07-01")

[*********************100%***********************]  1 of 1 completed
